In [16]:
import numpy as np
import pandas as pd

def simulate_per_sample(reference_df, libsize=12_000_000, nsim=5, pt_grid=None, sigma=0.25, interval=1.8):
    """
    Simulate RNA-seq count data for each sample in a reference dataset using
    gamma-Poisson (negative binomial) variation to model biological and technical noise.

    Parameters
    ----------
    reference_df : pandas.DataFrame
        Gene expression reference matrix with genes as rows and samples as columns.
        Values should be counts or expression measures proportional to counts.
    libsize : int, default=12_000_000
        Target total read count per simulated sample (library size).
    nsim : int, default=5
        Number of replicate simulations to generate per sample.
    pt_grid : array-like of float, optional
        Grid of proportion-transformation scaling factors to apply.
        If None, defaults to np.linspace(0.1, 1.0, 10).
    sigma : float, default=0.25
        Standard deviation of the log-normal noise term applied to biological coefficient
        of variation (BCV).
    interval : float, default=1.8
        Internal scaling factor used to derive the effective BCV multiplier.
        Together with the baseline constant, this results in an effective
        BCV scaling of 3.24 × pt.

    Returns
    -------
    dict of {float: pandas.DataFrame}
        Dictionary where keys are values from `pt_grid` and values are DataFrames
        of simulated counts. Each DataFrame has the same genes as rows and
        simulated replicate columns named `<sample>_sim<rep>`.

    Notes
    -----
    - The biological coefficient of variation (BCV) is computed as:

        BCV = (3.24 × pt) + 1 / sqrt(mu)

    with additional log-normal noise to introduce stochastic variability.
    - Counts are generated from a gamma–Poisson mixture, equivalent to a
    negative binomial distribution.
    - Zeros in the reference are replaced with small positive values to avoid issues
      in downstream computations.
    """
    if pt_grid is None:
        pt_grid = np.linspace(0.1, 1.0, 10)
    
    gene_names = reference_df.index
    samples = reference_df.columns
    
    all_simulations = {}
    
    for pt in pt_grid:
        scaled_pt = pt * interval
        sim_cols = []
        sim_data = []
        
        for sample in samples:
            # get single sample vector
            expr = reference_df[sample]
            
            # normalize to proportions
            r_i0 = expr / expr.sum()
            mu_i0 = r_i0 * libsize
            mu_i0 = mu_i0.replace(0, 1e-6)  # avoid zeros
            
            # replicate nsim times
            mu0 = np.tile(mu_i0.values.reshape(-1,1), (1, nsim))
            
            bcv0 = (1.8 * scaled_pt) + 1 / np.sqrt(mu0)
            delta = np.random.normal(0, sigma, mu0.shape)
            bcv = bcv0 * np.exp(delta / 2)
            
            shape = 1 / (bcv ** 2)
            scale = mu0 / shape
            mu = np.random.gamma(shape, scale)
            simulated_counts = np.random.poisson(mu)
            
            # prepare column names
            sim_cols += [f"{sample}_sim{i+1}" for i in range(nsim)]
            sim_data.append(simulated_counts)
        
        # concatenate all simulated samples for this pt
        sim_matrix = np.hstack(sim_data)
        df_sim = pd.DataFrame(sim_matrix, index=gene_names, columns=sim_cols)
        
        all_simulations[pt] = df_sim
    
    return all_simulations

In [18]:
reference_df = pd.read_csv("20250616_All-Tissues-NoDup_Counts_Clean.txt", delim_whitespace=True, index_col=0)

simulated_data = simulate_per_sample(reference_df, nsim=5)

for pt, df in simulated_data.items():
    filename = f"./20250616_All-Tissues-NoDup_Noise{pt:.1f}.txt"
    df.to_csv(filename, sep="\t")

/tmp/ipykernel_959729/2444847422.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  reference_df = pd.read_csv("20250616_All-Tissues-NoDup_Counts_Clean.txt", delim_whitespace=True, index_col=0)
